# 02 – Spatial unit

Choose the spatial unit for the analysis: points, or grid squares of 1 km, 2 km (tetrads) or 10 km (hectads).

The choice depends on how precise the record locations are. For NBN, `coordinateUncertaintyInMeters` shows this directly. For GBIF it is mostly missing, so precision has to be estimated in other ways.

Input: `data/processed/records.parquet`, built by `src/build_records.py` (merged and filtered GBIF + NBN records). Choices are logged in `decisions.md`.

In [ ]:
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import requests

RECORDS_PATH = "../data/processed/records.parquet"

## Load data

In [18]:
records = pd.read_parquet(RECORDS_PATH)

print(f"{len(records):,} records")  # expected: 433,993
records["source"].value_counts()

433,993 records


source
NBN     243079
GBIF    190914
Name: count, dtype: int64

In [19]:
# Column types: Parquet keeps dates as datetime and year/month as Int64
records.dtypes

source                                      str
record_id                                   str
occurrenceID                                str
dataset                                     str
eventDate                        datetime64[us]
year                                      Int64
month                                     Int64
decimalLatitude                         float64
decimalLongitude                        float64
coordinateUncertaintyInMeters           float64
gridReference                               str
stateProvince                               str
individualCount                         float64
vitality                                    str
countryCode                                 str
issue                                       str
dtype: object

In [20]:
records.head()

,source,record_id,occurrenceID,dataset,eventDate,year,month,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,gridReference,stateProvince,individualCount,vitality,countryCode,issue
0,NBN,40158fdb-f6cb-43b7-8842-59ba9a217a63,40158fdb-f6cb-43b7-8842-59ba9a217a63,Birds (BTO+partners),2013-01-08,2013,1,51.392379,1.426863,707.1,TR3871,England,NaN,alive,GB,NaN
1,NBN,06a1cc21-9e73-41d1-a31e-302f204da4d3,06a1cc21-9e73-41d1-a31e-302f204da4d3,Birds (BTO+partners),2021-03-23,2021,3,51.579580,-0.738633,707.1,SU8787,England,NaN,alive,GB,NaN
2,NBN,01a07a98-b7da-41ad-80b4-5f448f4502f3,01a07a98-b7da-41ad-80b4-5f448f4502f3,Birds (BTO+partners),2014-10-29,2014,10,51.277971,0.199401,707.1,TQ5355,England,NaN,alive,GB,NaN
3,NBN,aab3f43c-b2cb-4d48-b101-7d6c3ab41a8a,aab3f43c-b2cb-4d48-b101-7d6c3ab41a8a,Birds (BTO+partners),2019-06-05,2019,6,51.365927,-0.213000,707.1,TQ2464,England,NaN,alive,GB,NaN
4,NBN,057aa4ed-4c5f-4508-a2fe-a0f73b27ec74,057aa4ed-4c5f-4508-a2fe-a0f73b27ec74,Birds (BTO+partners),2016-11-05,2016,11,51.392379,1.426863,707.1,TR3871,England,NaN,alive,GB,NaN


## Location precision by source

Share of records with a coordinate uncertainty value, and the most common values.

In [21]:
# Share of records with an uncertainty value, per source
print(records.groupby("source")["coordinateUncertaintyInMeters"].apply(lambda s: s.notna().mean()).round(3))

# Most common uncertainty values per source
# (707.1 m = 1 km square, 1414.2 m = 2 km square, 7071.1 m = 10 km square: half the diagonal)
(records.groupby("source")["coordinateUncertaintyInMeters"]
        .value_counts()
        .groupby(level=0).head(8))

source
GBIF    0.057
NBN     1.000
Name: coordinateUncertaintyInMeters, dtype: float64


source  coordinateUncertaintyInMeters
GBIF    707.1                              5131
        4.0                                 476
        3.0                                 322
        8.0                                 299
        5.0                                 241
        15.0                                201
        1.0                                 171
        10.0                                152
NBN     707.1                            201186
        1414.2                            33982
        7071.1                             4882
        70.1                               2073
        70.7                                645
        7.1                                 132
        5.0                                  41
        50.0                                 25
Name: count, dtype: int64

### NBN: when are the 10 km records from?

Only 2% of NBN records have 10 km resolution, but they could matter more than that if they are concentrated in particular years.

In [22]:
# Share of NBN records per year with 10 km resolution (uncertainty 7071.1 m)
nbn = records[records["source"] == "NBN"]
is_10km = nbn["coordinateUncertaintyInMeters"] > 5000

share_10km = is_10km.groupby(nbn["year"]).mean()
share_10km[share_10km > 0].round(2)

year
1967    0.50
1968    0.75
1970    1.00
1971    1.00
1972    1.00
1973    0.93
1975    0.09
1976    0.33
1979    0.73
1980    0.30
1981    0.89
1984    0.08
1986    0.17
1987    0.31
1988    0.78
1989    0.08
1990    0.33
1991    0.09
1992    0.07
1993    0.04
1995    0.03
1996    0.02
1997    0.02
1998    0.06
1999    0.04
2000    0.01
2001    0.01
2002    0.00
2003    0.01
2004    0.00
2005    0.01
2006    0.02
2007    0.03
2008    0.03
2009    0.03
2010    0.02
2011    0.02
2012    0.06
2013    0.05
2014    0.04
2015    0.04
2016    0.04
2017    0.03
2018    0.02
2019    0.01
2020    0.01
2021    0.01
2022    0.01
2023    0.01
2024    0.01
2025    0.00
2026    0.02
Name: coordinateUncertaintyInMeters, dtype: float64

Until about 1990, a large share of NBN records (often more than half) have only 10 km resolution. From the early 1990s the share stays below 10%. Finer squares would drop much of the data for the start of the spread.

### GBIF: how concentrated are the locations?

GBIF records mostly have no uncertainty value. Most are eBird checklists, whose location is often a birding hotspot or the start of a walk. Many records at exactly the same coordinates point to hotspots.

In [23]:
# Count how often each GBIF location occurs (rounded to 5 decimals, ~1 m)
gbif = records[records["source"] == "GBIF"]
coords = gbif[["decimalLatitude", "decimalLongitude"]].round(5)
counts = coords.value_counts()

print(f"{len(counts):,} unique locations for {len(gbif):,} records")
print(f"Share of records at locations used 100+ times: {counts[counts >= 100].sum() / len(gbif):.0%}")

40,012 unique locations for 190,914 records
Share of records at locations used 100+ times: 54%


More than half of GBIF records come from locations used 100 times or more. At 2 km, the number of occupied squares would largely reflect where birders go; at 10 km a hotspot and the birds around it usually fall in the same square.

## Decision

- **Main unit: 10 km squares** (hectads, British National Grid). Every record fits, including NBN's 10 km records from the early years.
- **Comparison: 2 km squares** (tetrads), only for the later period when 10 km records are rare (from the early 1990s).

See `decisions.md`, "Spatial unit".

# Assign records to squares (try-out)

Convert each record's latitude/longitude to British National Grid (EPSG:27700) and round the easting and northing down to the square size. Each square gets an ID from its south-west corner in km, e.g. `530_180` is the 10 km square from 530–540 km east and 180–190 km north. NBN and GBIF records get the same kind of ID this way.

Once this works, the same step moves to `src/build_records.py`, so `records.parquet` includes the square IDs.

In [24]:
# Latitude/longitude (WGS84) -> British National Grid easting/northing in metres.
# Northern Ireland normally uses the Irish Grid; its ~65 records are placed on the
# British grid too, so all records use the same squares (see decisions.md).
points = gpd.points_from_xy(
    records["decimalLongitude"], records["decimalLatitude"], crs="EPSG:4326"
).to_crs("EPSG:27700")
records["easting"] = points.x
records["northing"] = points.y


def square_id(easting, northing, size_m):
    """ID of the grid square containing each point: south-west corner in km, e.g. '530_180'."""
    e_km = (easting // size_m * size_m / 1000).astype(int)
    n_km = (northing // size_m * size_m / 1000).astype(int)
    return e_km.astype(str) + "_" + n_km.astype(str)


records["hectad"] = square_id(records["easting"], records["northing"], 10_000)  # 10 km
records["tetrad"] = square_id(records["easting"], records["northing"], 2_000)   # 2 km

print(f"{records['hectad'].nunique():,} occupied 10 km squares")
print(f"{records['tetrad'].nunique():,} occupied 2 km squares")
records[["source", "decimalLatitude", "decimalLongitude", "easting", "northing", "gridReference", "hectad", "tetrad"]].head()

1,102 occupied 10 km squares
5,836 occupied 2 km squares


,source,decimalLatitude,decimalLongitude,easting,northing,gridReference,hectad,tetrad
0,NBN,51.392379,1.426863,638499.969834,171499.948280,TR3871,630_170,638_170
1,NBN,51.579580,-0.738633,487499.998184,187499.974161,SU8787,480_180,486_186
2,NBN,51.277971,0.199401,553499.982677,155500.024423,TQ5355,550_150,552_154
3,NBN,51.365927,-0.213000,524499.974794,164500.017199,TQ2464,520_160,524_164
4,NBN,51.392379,1.426863,638499.969834,171499.948280,TR3871,630_170,638_170


## Check 1: do NBN records land in the square of their own grid reference?

NBN records have an Ordnance Survey grid reference (e.g. `TQ2464`), which names the square directly. If the conversion from latitude/longitude is right, the 10 km square calculated from the coordinates should be the same as the one in the grid reference. Northern Ireland records may use Irish Grid references (one letter, e.g. `J3478`), which can't be compared this way and are skipped.

In [25]:
GRIDREF = re.compile(r"([A-HJ-Z]{2})(\d+)[A-NP-Z]?")  # two letters (no I), digits, optional tetrad letter
LETTERS = "ABCDEFGHJKLMNOPQRSTUVWXYZ"                 # grid letters skip I


def hectad_from_gridref(ref):
    """10 km square ID ('530_170') from a British OS grid reference, or None."""
    m = GRIDREF.fullmatch(ref) if isinstance(ref, str) else None
    if m is None or len(m.group(2)) % 2:
        return None
    l1, l2 = LETTERS.index(m.group(1)[0]), LETTERS.index(m.group(1)[1])
    e100 = ((l1 - 2) % 5) * 5 + (l2 % 5)        # 100 km square, easting
    n100 = (19 - (l1 // 5) * 5) - (l2 // 5)     # 100 km square, northing
    digits = m.group(2)
    half = len(digits) // 2
    e_km = e100 * 100 + int(digits[0]) * 10       # first digit of each half = 10 km square
    n_km = n100 * 100 + int(digits[half]) * 10
    return f"{e_km}_{n_km}"


nbn = records[records["source"] == "NBN"].copy()
nbn["hectad_gridref"] = nbn["gridReference"].map(hectad_from_gridref)

comparable = nbn["hectad_gridref"].notna()
match = nbn["hectad_gridref"] == nbn["hectad"]
print(f"Comparable: {comparable.sum():,} of {len(nbn):,} NBN records")
print(f"Same 10 km square: {match[comparable].mean():.2%}")

# Records that land in a different square, if any
nbn.loc[comparable & ~match, ["gridReference", "hectad_gridref", "hectad", "coordinateUncertaintyInMeters", "stateProvince"]].head(20)

Comparable: 243,038 of 243,079 NBN records
Same 10 km square: 100.00%


,gridReference,hectad_gridref,hectad,coordinateUncertaintyInMeters,stateProvince


## Check 2: map of occupied 10 km squares

Each occupied square drawn at its position, coloured by the first year with a record. Squares outside the UK, in the sea, or in an implausible pattern would point to a conversion problem.

In [ ]:
# One row per occupied 10 km square, with the first year it has a record
hectads = (
    records.groupby("hectad")
           .agg(first_year=("year", "min"), n_records=("record_id", "size"))
           .reset_index()
)
corner = hectads["hectad"].str.split("_", expand=True).astype(int)
hectads["e_km"] = corner[0] + 5   # square centre
hectads["n_km"] = corner[1] + 5

# Land outline: Natural Earth countries (public domain), downloaded once
LAND_URL = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
LAND_PATH = Path("../data/external/ne_10m_admin_0_countries.zip")
if not LAND_PATH.exists():
    LAND_PATH.parent.mkdir(parents=True, exist_ok=True)
    LAND_PATH.write_bytes(requests.get(LAND_URL, timeout=60).content)

countries = gpd.read_file(LAND_PATH)
land = (
    countries[countries["ADM0_A3"].isin(["GBR", "IRL", "IMN"])]  # UK, Ireland, Isle of Man
    .to_crs("EPSG:27700")
    .geometry.scale(xfact=0.001, yfact=0.001, origin=(0, 0))      # metres -> km, to match e_km/n_km
)

fig, ax = plt.subplots(figsize=(6, 9))
land.plot(ax=ax, color="#f0efec", edgecolor="#898781", linewidth=0.4, zorder=0)
sc = ax.scatter(hectads["e_km"], hectads["n_km"], c=hectads["first_year"].astype(float),
                cmap="Blues_r", marker="s", s=6, zorder=1)
fig.colorbar(sc, ax=ax, label="First year with a record", shrink=0.6)

# Zoom to the occupied squares (otherwise all of Ireland is shown)
ax.set_xlim(hectads["e_km"].min() - 30, hectads["e_km"].max() + 30)
ax.set_ylim(hectads["n_km"].min() - 30, hectads["n_km"].max() + 30)
ax.set_aspect("equal")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
ax.set_title("Occupied 10 km squares")
plt.tight_layout()

print(f"Easting range: {hectads['e_km'].min()}–{hectads['e_km'].max()} km")
print(f"Northing range: {hectads['n_km'].min()}–{hectads['n_km'].max()} km")